In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## దశ 1: వ్యవస్థబద్ధీకృత అవుట్‌పుట్‌ల కోసం Pydantic నమూనాలు నిర్వచించండి

ఈ నమూనాలు ఏజెంట్లు తిరిగిచ్చే **స్కీము** ని నిర్వచిస్తాయి. Pydantic తో `response_format` ఉపయోగించడం ద్వారా ఇము:
- ✅ రకం-భద్రత గల డేటా సేకరణ
- ✅ ఆటోమేటిక్ ధృవీకరణ
- ✅ ఉచ్ఛారిత టెక్స్ట్ స్పందనల నుండి వాచిక దోషాలు లేవు
- ✅ ఫీల్డుల ఆధారంగా సులభమైన షరతు రూటింగ్


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## దశ 2: హోటల్ బుకింగ్ టూల్‌ని సృష్టించండి

ఈ టూల్‌ను **availability_agent** గదులు అందుబాటులో ఉన్నాయా అని తనిఖీ చేయడానికి పిలవబోతుంది. మేము `@ai_function` డెకరేటర్‌ను ఉపయోగిస్తాము:
- ఒక పython ఫంక్షన్‌ని AI-పిలిచే సాధనంగా మార్చడానికి
- LLM కోసం JSON స్కీమాను ఆటోమేటిగ్గా ఉత్పత్తి చేయడానికి
- పారామీటర్ చెల్లుబాటు తనిఖీ చేయడానికి
- ఏజెంట్లు ఆటోమేటిక్‌గా పిలవటం సదుపాయాన్ని కల్పించడానికి

ఈ డెమో కోసం:
- **స్టాక్‌హోమ్, సియాటెల్, టోక్యో, లండన్, ఆంస్టర్డామ్** → గదులు ఉన్నాయి ✅
- **ఇతర అన్ని నగరాలు** → గదులు లేవు ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## దశ 3: రౌటింగ్ కోసం షరతు ఫంక్షన్లను నిర్వచించండి

ఈ ఫంక్షన్లు ఏజెంట్ యొక్క ప్రతిస్పందనను పరిశీలించి, వర్క్‌ఫ్లోలో ఏ మార్గాన్ని అనుసరించాలో నిర్ణయిస్తాయి.

**ప్రధాన నమూనా:**
1. సందేశం `AgentExecutorResponse`నా అని పరీక్షించండి
2. నిర్మాణబద్ధమైన అవుట్‌పుట్‌ను (Pydantic మోడల్) పార్స్ చేయండి
3. రౌటింగ్‌ను నియంత్రించడానికి `True` లేదా `False`ను తిరిగి ఇవ్వండి

తదుపరి ఎగ్జిక్యూటర్‌ను పిలవడానికి ఈ షరతులను **ఎడ్జ్‌లపై** వర్క్‌ఫ్లో పరిశీలిస్తుంది.


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## దశ 4: కస్టమ్ డిస్ప్లే Executor సృష్టించండి

Executors అనేవి వర్క్‌ఫ్లో భాగాలు, అవి మార్పులు లేదా సైడ్ ఎఫెక్ట్స్‌ను నిర్వహిస్తాయి. మేము చివరి ఫలితాన్ని చూపించడానికి `@executor` డెకొరేటర్‌ని ఉపయోగించి ఒక కస్టమ్ executor సృష్టిస్తాము.

**ప్రధాన కాన్సెప్ట్‌లు:**
- `@executor(id="...")` - ఒక ఫంక్షన్‌ను వర్క్‌ఫ్లో executor గా నమోదుచేస్తుంది
- `WorkflowContext[Never, str]` - ఇన్‌పుట్/ఔట్‌పుట్ కోసం టైప్ సూచనలు
- `ctx.yield_output(...)` - తుది వర్క్‌ఫ్లో ఫలితాన్ని ఇస్తుంది


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## దశ 5: పర్యావరణ వేరియబుల్స్ లోడ్ చేయండి

LLM క్లయింట్‌ని కాన్ఫిగర్ చేయండి. ఈ ఉదాహరణ ఈ క్రింది వాటితో పనిచేస్తుంది:
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — సిఫార్సు చేయబడింది)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## దశ 6: నిర్మిత అవుట్పుట్లతో AI ఏజెంట్లను సృష్టించండి

మేము **మూడు ప్రత్యేక ఏజెంట్లను** సృష్టిస్తాము, ప్రతి ఒక్కటింటిని ఒక `AgentExecutor`లో చుట్టబడినది:

1. **availability_agent** - టూల్ ఉపయోగించి హోటల్ అందుబాటును తనిఖీ చేస్తుంది
2. **alternative_agent** - ప్రత్యామ్నాయ నగరాలను సూచిస్తుంది (పగడ్లు లేనప్పుడు)
3. **booking_agent** - బుకింగ్ ప్రోత్సహిస్తుంది (పగడ్లు ఉన్నప్పుడు)

**ముఖ్య లక్షణాలు:**
- `tools=[hotel_booking]` - ఏజెంట్కు టూల్ అందిస్తుంది
- `response_format=PydanticModel` - నిర్మిత JSON అవుట్పుట్‌ను బలవంతంగా రూపొందిస్తుంది
- `AgentExecutor(..., id="...")` - వర్క్‌ఫ్లో ఉపయోగం కోసం ఏజెంట్ను చుట్టుతుంది


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## దశ 7: శరతుల ఆధారంగా ఎడ్జ్‌లతో వర్క్‌ఫ్లోని నిర్మించండి

ఇప్పుడు మనం `WorkflowBuilder` ను ఉపయోగించి షరతులతో కూడిన రూటింగ్‌తో గ్రాఫ్‌ను నిర్మిస్తాము:

**వర్క్‌ఫ్లో నిర్మాణం:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**ప్రధాన పద్ధతులు:**
- `.set_start_executor(...)` - ప్రవేశ పాయింట్‌ని సెట్ చేస్తుంది
- `.add_edge(from, to, condition=...)` - షరతు ఆధారిత ఎడ్జ్‌ని జోడిస్తుంది
- `.build()` - వర్క్‌ఫ్లోని పూర్తిచేస్తుంది


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## దశ 8: టెస్ట్ కేస్ 1 ను నడపండి - నగరం ఎవరైతే అందుబాటులో లేదు (పారిస్)

మన సిమ్యులేషన్‌లో రూములు లేని పారిస్‌లో హోటల్స్ అభ్యర్థించడం ద్వారా **ఏమీ అందుబాటు లేదు** మార్గాన్ని పరీక్షిద్దాం.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## దశ 9: టెస్ట్ కేస్ 2 ను నడుపండి - నగరం WITH లభ్యత (స్టాక్‌హోమ్)

ఇప్పుడు మన సిమ్యులేషన్‌లో గదులున్న స్టాక్‌హోమ్ లోని హోటళ్లను అభ్యర్థించడం ద్వారా **లభ్యత** మార్గాన్ని పరీక్షిద్దాం.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## ముఖ్యమైన విషయాలు మరియు తదుపరి దశలు

### ✅ మీరు నేర్చుకున్నది:

1. **WorkflowBuilder ప్యాటర్న్**
   - ప్రవేశ బిందువు నిర్వచించడానికి `.set_start_executor()` ఉపయోగించండి
   - షరతు రూటింగ్ కోసం `.add_edge(from, to, condition=...)` ఉపయోగించండి
   - వర్క్‌ఫ్లోని ముగించడానికి `.build()` పిలవండి

2. **షరతు రూటింగ్**
   - షరతు ఫంక్షన్‌లు `AgentExecutorResponse`ని పరిశీలిస్తాయి
   - రూటింగ్ నిర్ణయాలను తీసుకోవడానికి నిర్మాణాత్మక అవుట్‌పుట్‌లు విశ్లేషించండి
   - ఒక ఎడ్జ్‌ను సక్రియం చేయడానికి `True` తిరిగి ఇవ్వాలి, దాన్ని దాటవేయడానికి `False`

3. **టూల్ ఇంటిగ్రేషన్**
   - Python ఫంక్షన్‌లను AI టూల్స్‌గా మార్చడానికి `@ai_function` ఉపయోగించండి
   - ఏజెంట్‌లు అవసరమైతే ఆటోమేటిక్‌గా టూల్స్‌ను పిలుస్తాయి
   - టూల్స్ JSON ని తిరిగి ఇస్తాయి, దాన్ని ఏజెంట్‌లు పార్స్ చేయవచ్చు

4. **నిర్మాణాత్మక అవుట్‌పుట్‌లు**
   - టైప్-సేఫ్ డేటా ఎక్స్‌ట్రాక్షన్ కోసం Pydantic మోడల్స్ ఉపయోగించండి
   - ఏజెంట్‌లను సృష్టించేటప్పుడు `response_format=MyModel` సెట్చేయండి
   - `Model.model_validate_json()` తో ప్రతిస్పందనలను పార్స్ చేయండి

5. **కస్టమ్ ఎగ్జిక్యూటర్లు**
   - వర్క్‌ఫ్లో భాగాలు సృష్టించడానికి `@executor(id="...")` ఉపయోగించండి
   - ఎగ్జిక్యూటర్లు డేటాను మార్చగలవు లేదా సైడ్-ఎఫెక్ట్స్ చేయగలవు
   - వర్క్‌ఫ్లో ఫలితాలను ఉత్పత్తి చేయడానికి `ctx.yield_output()` ఉపయోగించండి

### 🚀 యథార్థ ప్రపంచ అనువర్తనాలు:

- **ప్రయాణ బుకింగ్**: అందుబాటును తనిఖీ చేయండి, ప్రత్యామ్నాయాలు సూచించండి, ఎంపికలను పోల్చండి
- **కస్టమర్ సర్వీస్**: సమస్య రకం, భావోద్వేగం, ప్రాధాన్యత ఆధారంగా రూట్ చేయండి
- **ఈ-కామర్స్**: నిల్వను తనిఖీ చేయండి, ప్రత్యామ్నాయాలు సూచించండి, ఆర్డర్లను ప్రాసెస్ చేయండి
- **కంటెంట్ మోడరేషన్**: విషపు స్కోర్లు, యూజర్ ఫ్లాగ్స్ ఆధారంగా రూటింగ్ చేయండి
- **అంగీకార వర్క్‌ఫ్లోలు**: మొత్తం, యూజర్ పాత్ర, ప్రమాద స్థాయి ఆధారంగా రూట్ చేయండి
- **బహుస్థాయి ప్రాసెసింగ్**: డేటా నాణ్యత, పూర్తి స్థితి ఆధారంగా రూట్ చేయండి

### 📚 తదుపరి దశలు:

- మరిన్ని సంక్లిష్ట షరతులు చేర్చండి (బహుళ ప్రమాణాలు)
- వర్క్‌ఫ్లో స్థితి నిర్వహణతో లూప్‌లను అమలు చేయండి
- పునర్వినియోగపరచదగిన భాగాలకు ఉప-వర్క్‌ఫ్లోలను జోడించండి
- వాస్తవ APIs (హోటల్ బుకింగ్, నిల్వ వ్యవస్థలతో) ఇంటిగ్రేట్ చేయండి
- దోష నిర్వహణ మరియు బ్యాకప్ మార్గాలను జోడించండి
- నిర్మించబడిన విజువలైజేషన్ టూల్స్‌తో వర్క్‌ఫ్లోలను చూపించండి


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
